# AI Automated BI Dashboard + Executive Summary

# Phase 1 - Calculate the KPIs

There are six KPIs needed to be calculated.
1. Monthly revenue trend
2. Revenue by product category
3. Average order value by month
4. Delivery time vs review score
5. Revenue by customer state
6. Period-over-period comparison

In [2]:
# Cell 1 - Import the required libraries

import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from google import genai
from google.genai import types
import prettytable
# Manually restore the missing constant
prettytable.DEFAULT = 'DEFAULT'
prettytable.DEFAULT = prettytable._DEPRECATED_DEFAULT
prettytable.PLAIN_COLUMNS = prettytable._DEPRECATED_PLAIN_COLUMNS
prettytable.MSWORD_FRIENDLY = prettytable._DEPRECATED_MSWORD_FRIENDLY
prettytable.RANDOM = prettytable._DEPRECATED_RANDOM


In [58]:
from sqlalchemy import URL

load_dotenv()
os.environ["DATABASE_URL"] = URL.create(
    "postgresql+psycopg",
    username=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD"),
    host=os.getenv("PGHOST", "localhost"),
    port=int(os.getenv("PGPORT", 5432)),
    database=os.getenv("PGDATABASE", "olist"),
).render_as_string(hide_password=False)

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.displaycon = False

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [4]:
%config SqlMagic.autopandas  = True    # return DataFrames, not ResultSets
%config SqlMagic.displaylimit = 20     # display cap; full result is still stored
%config SqlMagic.displaycon  = False   # don't echo the connection after each cell
%config SqlMagic.feedback    = False   # suppress "Returning data to local variable"

In [59]:
%%sql monthly <<
SELECT date_trunc('month', o.order_purchase_timestamp)::date     AS month,
       SUM(oi.price + oi.freight_value)                          AS revenue,
       COUNT(DISTINCT o.order_id)                                AS orders,
       SUM(oi.price + oi.freight_value)
         / COUNT(DISTINCT o.order_id)                            AS aov
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY 1;


Returning data to local variable monthly


In [6]:
monthly.head(10)

,month,revenue,orders,aov
0,2016-09-01,143.46,1,143.4600000000000000
1,2016-10-01,46490.66,265,175.4364528301886792
2,2016-12-01,19.62,1,19.6200000000000000
3,2017-01-01,127482.37,750,169.9764933333333333
4,2017-02-01,271239.32,1653,164.0891228070175439
5,2017-03-01,414330.95,2546,162.7380007855459544
6,2017-04-01,390812.40,2303,169.6970907511940947
7,2017-05-01,566851.40,3546,159.8565707839819515
8,2017-06-01,490050.37,3135,156.3159074960127592
9,2017-07-01,566299.08,3872,146.2549276859504132


In [7]:
%%sql category <<
SELECT COALESCE(t.product_category_name_english,
                p.product_category_name, 'unknown')              AS category,
       SUM(oi.price + oi.freight_value)                          AS revenue,
       COUNT(DISTINCT o.order_id)                                AS orders
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN products    p  ON p.product_id = oi.product_id
LEFT JOIN product_category_name_translation t
       ON t.product_category_name = p.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY revenue DESC
LIMIT 10;


Returning data to local variable category


In [8]:
%%sql delivery <<
WITH delivered AS (
    SELECT o.order_id,
           (o.order_delivered_customer_date::date
              - o.order_purchase_timestamp::date) AS delivery_days
    FROM orders o
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
)
SELECT CASE WHEN d.delivery_days <=  3 THEN '0-3 days'
            WHEN d.delivery_days <=  7 THEN '4-7 days'
            WHEN d.delivery_days <= 14 THEN '8-14 days'
            WHEN d.delivery_days <= 30 THEN '15-30 days'
            ELSE '30+ days' END                   AS delivery_bucket,
       MIN(d.delivery_days)                       AS sort_key,
       COUNT(*)                                   AS orders,
       ROUND(AVG(r.review_score), 2)              AS avg_review_score
FROM delivered d
JOIN order_reviews r ON r.order_id = d.order_id
GROUP BY 1
ORDER BY sort_key;


Returning data to local variable delivery


In [9]:
%%sql state <<
SELECT c.customer_state                                          AS state,
       SUM(oi.price + oi.freight_value)                          AS revenue,
       COUNT(DISTINCT o.order_id)                                AS orders
FROM orders o
JOIN customers   c  ON c.customer_id = o.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY revenue DESC;


Returning data to local variable state


In [10]:
%%sql periods <<
WITH bounds AS (
    SELECT date_trunc('month', MAX(order_purchase_timestamp)) AS partial_month
    FROM orders WHERE order_status = 'delivered'
)
SELECT to_char(partial_month - INTERVAL '1 month',  'FMMonth YYYY') AS current_period,
       to_char(partial_month - INTERVAL '2 months', 'FMMonth YYYY') AS previous_period
FROM bounds;


Returning data to local variable periods


In [11]:
%%sql raw_cmp <<
WITH bounds AS (
    SELECT date_trunc('month', MAX(order_purchase_timestamp)) AS partial_month
    FROM orders WHERE order_status = 'delivered'
),
periods AS (
    SELECT partial_month - INTERVAL '1 month'  AS curr_start,
           partial_month                       AS curr_end,
           partial_month - INTERVAL '2 months' AS prev_start
    FROM bounds
),
order_facts AS (                     -- one row per order, not per line item
    SELECT CASE WHEN o.order_purchase_timestamp >= p.curr_start
                THEN 'current' ELSE 'previous' END               AS period,
           (SELECT SUM(oi.price + oi.freight_value)
              FROM order_items oi WHERE oi.order_id = o.order_id) AS order_revenue,
           (o.order_delivered_customer_date::date
              - o.order_purchase_timestamp::date)                AS delivery_days,
           (SELECT AVG(r.review_score)
              FROM order_reviews r WHERE r.order_id = o.order_id) AS review_score
    FROM orders o
    CROSS JOIN periods p
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= p.prev_start
      AND o.order_purchase_timestamp <  p.curr_end
)
SELECT period,
       SUM(order_revenue)  AS revenue,
       COUNT(*)            AS orders,
       AVG(order_revenue)  AS aov,
       AVG(delivery_days)  AS avg_delivery_days,
       AVG(review_score)   AS avg_review_score
FROM order_facts
GROUP BY 1;


Returning data to local variable raw_cmp


In [12]:
pr = periods.iloc[0]          # 1-row DataFrame → Series


def comparison_table(df):
    wide = df.set_index("period").T.astype(float)
    wide["pct_change"] = (wide["current"] - wide["previous"]) / wide["previous"] * 100
    return wide.reset_index(names="metric").round(2)


comparison = comparison_table(raw_cmp)
print(f"{pr.current_period} vs {pr.previous_period}")
print(comparison.to_markdown(index=False))   # ← the string Phase 4 sends to Gemini


July 2018 vs June 2018
| metric            |        current |       previous |   pct_change |
|:------------------|---------------:|---------------:|-------------:|
| revenue           |    1.02781e+06 |    1.01198e+06 |         1.56 |
| orders            | 6159           | 6099           |         0.98 |
| aov               |  166.88        |  165.93        |         0.57 |
| avg_delivery_days |    8.89        |    9.16        |        -3.02 |
| avg_review_score  |    4.32        |    4.31        |         0.26 |


In [13]:
CURRENCY = "R$"   # Olist is a Brazilian marketplace — say so explicitly

FMT = {
    "revenue":           lambda v: f"{CURRENCY} {v:,.0f}",
    "orders":            lambda v: f"{v:,.0f}",
    "aov":               lambda v: f"{CURRENCY} {v:,.2f}",
    "avg_delivery_days": lambda v: f"{v:.2f} days",
    "avg_review_score":  lambda v: f"{v:.2f} / 5",
}

def to_prompt_table(df):
    default = lambda v: f"{v:,.2f}"
    rows = [{
        "metric":   r["metric"],
        "current":  FMT.get(r["metric"], default)(r["current"]),
        "previous": FMT.get(r["metric"], default)(r["previous"]),
        "change":   f"{r['pct_change']:+.2f}%",
    } for _, r in df.iterrows()]
    return pd.DataFrame(rows).to_markdown(index=False)

metrics_md = to_prompt_table(comparison)
print(metrics_md)

| metric            | current      | previous     | change   |
|:------------------|:-------------|:-------------|:---------|
| revenue           | R$ 1,027,807 | R$ 1,011,978 | +1.56%   |
| orders            | 6,159        | 6,099        | +0.98%   |
| aov               | R$ 166.88    | R$ 165.93    | +0.57%   |
| avg_delivery_days | 8.89 days    | 9.16 days    | -3.02%   |
| avg_review_score  | 4.32 / 5     | 4.31 / 5     | +0.26%   |


In [14]:
# Load .env
load_dotenv(override=True)

# Read ONLY the Gemini key we intend to use
gemini_api_key = os.getenv("GEMINI_API_KEY")

if not gemini_api_key:
    raise RuntimeError(
        "GEMINI_API_KEY was not found in .env"
    )

client = genai.Client(api_key=gemini_api_key)          # picks the key up from the environment

for m in client.models.list():
    if "flash" in m.name.lower() and "generateContent" in (m.supported_actions or []):
        print(m.name)


models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview


In [15]:
MODEL_ID = "models/gemini-3.6-flash"

In [16]:
SYSTEM = (
    "You are a senior e-commerce analyst writing a monthly executive summary for the "
    "founder of a Brazilian online marketplace. Plain English, no jargon. You never "
    "state a number that does not appear in the data you are given, and you never "
    "speculate beyond what the metrics support."
)

def build_prompt(period_now, period_prev, table_md):
    return f"""Month-over-month performance for a Brazilian e-commerce marketplace.
Reporting period: {period_now}, compared against {period_prev}.
All monetary values are Brazilian reais (BRL).

{table_md}

Write exactly 3-4 sentences covering:
1. What changed most materially this month.
2. The most plausible driver, reasoning only from the relationships between the metrics above.
3. One specific recommended action.

Rules:
- Use only numbers that appear in the table. Do not compute new figures.
- Plain prose only: no bullet points, no headings.
- Do not open with a greeting or restate the reporting period."""

prompt = build_prompt(pr.current_period, pr.previous_period, metrics_md)
print(prompt)


Month-over-month performance for a Brazilian e-commerce marketplace.
Reporting period: July 2018, compared against June 2018.
All monetary values are Brazilian reais (BRL).

| metric            | current      | previous     | change   |
|:------------------|:-------------|:-------------|:---------|
| revenue           | R$ 1,027,807 | R$ 1,011,978 | +1.56%   |
| orders            | 6,159        | 6,099        | +0.98%   |
| aov               | R$ 166.88    | R$ 165.93    | +0.57%   |
| avg_delivery_days | 8.89 days    | 9.16 days    | -3.02%   |
| avg_review_score  | 4.32 / 5     | 4.31 / 5     | +0.26%   |

Write exactly 3-4 sentences covering:
1. What changed most materially this month.
2. The most plausible driver, reasoning only from the relationships between the metrics above.
3. One specific recommended action.

Rules:
- Use only numbers that appear in the table. Do not compute new figures.
- Plain prose only: no bullet points, no headings.
- Do not open with a greeting or restat

In [17]:
resp = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
            config=types.GenerateContentConfig(
            system_instruction=SYSTEM,
            temperature=0.3,
            max_output_tokens=1500,
            thinking_config=types.ThinkingConfig(
                thinking_level=types.ThinkingLevel.MINIMAL
            ),
        ),

)

fr = resp.candidates[0].finish_reason
if fr.name == "MAX_TOKENS":
    u = resp.usage_metadata
    raise RuntimeError(
        f"Truncated: {u.thoughts_token_count} thinking + "
        f"{u.candidates_token_count} output hit the cap. Raise max_output_tokens."
    )
if not resp.text:
    raise RuntimeError(f"No text returned (finish_reason={fr})")

summary_text = resp.text.strip()
print(summary_text)
print("\n—", resp.model_version, "|", resp.usage_metadata.total_token_count, "tokens")


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


The most material operational change this month was a -3.02% decrease in average delivery time, dropping from 9.16 days to 8.89 days. This faster fulfillment likely helped support minor increases in both customer feedback and basket size, with average review scores ticking up from 4.31 / 5 to 4.32 / 5 (+0.26%) and average order value rising from R$ 165.93 to R$ 166.88 (+0.57%). Together with a slight increase in order volume from 6,099 to 6,159 (+0.98%), overall revenue grew from R$ 1,011,978 to R$ 1,027,807 (+1.56%). To keep expanding overall revenue, you should focus efforts on logistics strategies that maintain delivery times below 8.89 days while finding ways to boost order volume past the recent +0.98% growth rate.

— gemini-3.6-flash | 617 tokens


In [18]:
print(repr(summary_text))
print("chars:", len(summary_text))
print("finish_reason:", resp.candidates[0].finish_reason)
print(resp.usage_metadata)


'The most material operational change this month was a -3.02% decrease in average delivery time, dropping from 9.16 days to 8.89 days. This faster fulfillment likely helped support minor increases in both customer feedback and basket size, with average review scores ticking up from 4.31 / 5 to 4.32 / 5 (+0.26%) and average order value rising from R$ 165.93 to R$ 166.88 (+0.57%). Together with a slight increase in order volume from 6,099 to 6,159 (+0.98%), overall revenue grew from R$ 1,011,978 to R$ 1,027,807 (+1.56%). To keep expanding overall revenue, you should focus efforts on logistics strategies that maintain delivery times below 8.89 days while finding ways to boost order volume past the recent +0.98% growth rate.'
chars: 729
finish_reason: FinishReason.STOP
cache_tokens_details=None cached_content_token_count=None candidates_token_count=220 candidates_tokens_details=None prompt_token_count=397 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
 

In [19]:
print(metrics_md)


| metric            | current      | previous     | change   |
|:------------------|:-------------|:-------------|:---------|
| revenue           | R$ 1,027,807 | R$ 1,011,978 | +1.56%   |
| orders            | 6,159        | 6,099        | +0.98%   |
| aov               | R$ 166.88    | R$ 165.93    | +0.57%   |
| avg_delivery_days | 8.89 days    | 9.16 days    | -3.02%   |
| avg_review_score  | 4.32 / 5     | 4.31 / 5     | +0.26%   |


In [20]:
import re

pat = r"\d[\d,]*\.?\d*"
in_summary = set(re.findall(pat, summary_text))
in_table   = set(re.findall(pat, metrics_md))
suspect    = {n for n in in_summary - in_table if len(n) > 2}

print("Numbers not traceable to the source table:", suspect or "none")


Numbers not traceable to the source table: none


In [21]:
%%sql
CREATE TABLE IF NOT EXISTS exec_summary (
    id           SERIAL PRIMARY KEY,
    generated_at TIMESTAMP NOT NULL DEFAULT now(),
    period_label TEXT      NOT NULL,
    summary_text TEXT      NOT NULL,
    model_used   TEXT
);


""


In [22]:
period_label = pr.current_period      # "July 2018"
model_used   = resp.model_version
# summary_text already exists from Phase 4


In [23]:
%%sql
DELETE FROM exec_summary WHERE length(summary_text) < 100;


""


In [24]:
%%sql
INSERT INTO exec_summary (period_label, summary_text, model_used)
VALUES (:period_label, :summary_text, :model_used);


""


In [25]:
%%sql
SELECT count(*) AS rows, max(generated_at) AS latest FROM exec_summary;


,rows,latest
0,2,2026-09-12 13:58:41.129467


In [26]:
%%sql
SELECT id, generated_at, period_label, model_used,
       left(summary_text, 80) || '...' AS preview
FROM exec_summary
ORDER BY generated_at DESC
LIMIT 5;


,id,generated_at,period_label,model_used,preview
0,5,2026-09-12 13:58:41.129467,July 2018,gemini-3.6-flash,The most material operational change this mont...
1,4,2026-09-10 04:43:48.366066,July 2018,gemini-3.6-flash,The most material change this month was a -3.0...


In [27]:
%%sql sim <<
WITH o AS (
    SELECT o.order_id,
           date_trunc('month', o.order_purchase_timestamp)::date          AS month,
           (o.order_delivered_customer_date::date
              - o.order_purchase_timestamp::date)                        AS dd,
           (SELECT SUM(price + freight_value) FROM order_items i
             WHERE i.order_id = o.order_id)                               AS rev,
           (SELECT AVG(review_score) FROM order_reviews r
             WHERE r.order_id = o.order_id)                               AS rs
    FROM orders o
    WHERE o.order_status = 'delivered'
)
SELECT month     AS "Month Start",
       SUM(rev)  AS "Revenue",
       COUNT(*)  AS "Orders",
       AVG(rev)  AS "AOV",
       AVG(rs)   AS "Avg Review Score",
       AVG(dd)   AS "Avg Delivery Days"
FROM o GROUP BY 1 ORDER BY 1;


Returning data to local variable sim


In [29]:
dataset = sim.copy()
dataset["Selected Category"] = "All categories"
dataset["Selected State"]    = "All states"

In [28]:
from pathlib import Path
from datetime import datetime

out_dir = Path("outputs"); out_dir.mkdir(exist_ok=True)
slug = period_label.replace(" ", "-").lower()
path = out_dir / f"summary_{slug}.md"

path.write_text(
    f"# Executive summary — {period_label}\n\n"
    f"{summary_text}\n\n---\n\n"
    f"*Generated {datetime.now():%Y-%m-%d %H:%M} by `{model_used}` "
    f"from Olist marketplace data.*\n\n"
    f"{metrics_md}\n",
    encoding="utf-8",
)
print("wrote", path.resolve())


wrote C:\Users\manas\OneDrive\Desktop\Projects - AI + BI\outputs\summary_july-2018.md
